
# Exp.05B — Residual Re-encoding After Pose Editing

Exp.05 compared two inference policies after a pose-code edit:

- **pose_only**: discard residual information after editing;
- **retain_residual**: keep the residual codes extracted from the original motion.

Exp.05B adds the third policy requested by the reviewer discussion:

- **reencode_residual**: after changing the pose code, recompute the residual representation
  from the original motion **conditioned on the edited pose code**, then decode
  `edited pose + re-encoded residual`.

No model is retrained.

## Why this experiment matters

The tokenizer decomposes the motion latent into

\[
z \approx p(\text{pose code}) + r.
\]

If the pose code is changed but the original residual is retained, the residual may partly
conflict with the edit. If the residual is re-encoded against the edited pose code, the new
residual may instead compensate for the pose change and reconstruct the original motion,
which could improve preservation but weaken edit effectiveness.

Therefore this experiment directly tests the three practical policies:

1. drop the residual;
2. retain the pre-edit residual;
3. re-encode the residual after editing.

## Exact pairing with Exp.05

Exp.05B **does not resample data**. It reuses:

- the same 400 HumanML3D test motions,
- the same target body part,
- the same temporal edit window,
- the same target pose-code category,

from the completed Exp.05 production CSV.

This makes all policy comparisons paired at the motion level.


In [ ]:

from pathlib import Path
import os, sys, json, hashlib, subprocess, shlex, time

PGR = Path(".")
PY = Path("venv/bin/python")
DATA = PGR / "dataset" / "HumanML3D"
PRE = PGR / "revision_preflight"

EXP05 = PGR / "revision_results" / "exp05_editing" / "production"
RESULTS = PGR / "revision_results" / "exp05b_reencode"
LOGDIR = PGR / "revision_logs" / "exp05b_reencode"

EXP05_ROWS = EXP05 / "exp05_per_sample.csv"
EXP05_AUDIT = EXP05 / "exp05_completion_audit.json"
EXP05_SELECTED = EXP05 / "exp05_selected_samples.json"
EXP04B_AUDIT = PGR / "revision_results" / "exp04b_refine" / "exp04b_refine_completion_audit.json"

SCRIPT = PRE / "run_exp05b_reencode.py"
LOG = LOGDIR / "exp05b_reencode.log"
PIDFILE = LOGDIR / "exp05b_reencode.pid"
LAUNCH = LOGDIR / "exp05b_reencode_launch.json"

for d in [PRE, RESULTS, LOGDIR]:
    d.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

required = [
    PY, PGR, DATA,
    EXP05_ROWS, EXP05_AUDIT, EXP05_SELECTED, EXP04B_AUDIT,
    DATA / "new_joint_vecs",
    DATA / "codes",
    DATA / "Mean.npy",
    DATA / "Std.npy",
]
for p in required:
    assert p.exists(), p

exp05_audit = json.loads(EXP05_AUDIT.read_text())
assert int(exp05_audit["completed_samples"]) == 400, exp05_audit

b = json.loads(EXP04B_AUDIT.read_text())
TOKENIZER_DIR = Path(b["tokenizer_dir"])
TOKENIZER_CKPT = TOKENIZER_DIR / "net_best_fid.pth"

assert TOKENIZER_CKPT.exists(), TOKENIZER_CKPT
assert (TOKENIZER_DIR / "arguments.yaml").exists()

print("=== EXP.05B PRE-FLIGHT ===")
print("Python       :", sys.executable)
print("Exp.05 rows  :", EXP05_ROWS)
print("Exp.05 audit :", EXP05_AUDIT)
print("Tokenizer    :", TOKENIZER_CKPT)
print("Tokenizer SHA256:", sha256(TOKENIZER_CKPT))
print("[OK] Exp.05 completed data and tokenizer resolved.")


In [ ]:
# Build the standalone Exp.05B experiment script.
script_text = '\nimport os, sys, json, time, random, hashlib, argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport yaml\n\nPGR = Path(".")\nos.chdir(PGR)\nsys.path.insert(0, str(PGR))\n\nif not hasattr(np, "float"):\n    np.float = float\nif not hasattr(np, "int"):\n    np.int = int\nif not hasattr(np, "bool"):\n    np.bool = bool\n\nfrom models.pg_tokenizer import PoseGuidedTokenizer\nfrom utils.motion_process import recover_from_ric\nfrom utils.codebook import vq_to_range\n\nTARGETS = {\n    "left_knee":  dict(group_id=0, triplet=(1,4,7), target_joints=(1,4,7)),\n    "right_knee": dict(group_id=1, triplet=(2,5,8), target_joints=(2,5,8)),\n    "left_elbow": dict(group_id=2, triplet=(16,18,20), target_joints=(16,18,20)),\n    "right_elbow":dict(group_id=3, triplet=(17,19,21), target_joints=(17,19,21)),\n}\nANGLE_THRESHOLDS = np.arange(10, 180, 10, dtype=np.float32)\n\ndef fixseed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1024*1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\ndef build_tokenizer(folder):\n    import argparse as _argparse\n    folder = Path(folder)\n    a = _argparse.Namespace(**yaml.safe_load((folder / "arguments.yaml").read_text()))\n    net = PoseGuidedTokenizer(\n        a,\n        a.nb_code,\n        a.code_dim,\n        a.output_emb_width,\n        a.down_t,\n        a.stride_t,\n        a.width,\n        a.depth,\n        a.dilation_growth_rate,\n        a.vq_act,\n        a.vq_norm,\n        num_quantizers=a.rvq_num_quantizers,\n        shared_codebook=a.rvq_shared_codebook,\n        quantize_dropout_prob=a.rvq_quantize_dropout_prob,\n        quantize_dropout_cutoff_index=a.rvq_quantize_dropout_cutoff_index,\n        rvq_nb_code=a.rvq_nb_code,\n        mu=a.rvq_mu,\n        residual_ratio=a.rvq_residual_ratio,\n        vq_loss_beta=a.rvq_vq_loss_beta,\n        quantizer_type=a.rvq_quantizer_type,\n        params_soft_ent_loss=a.params_soft_ent_loss,\n        use_ema=(not a.unuse_ema),\n        init_method=a.rvq_init_method,\n    )\n    ckpt_path = folder / "net_best_fid.pth"\n    ckpt = torch.load(ckpt_path, map_location="cpu")\n    net.load_state_dict(ckpt["net"], strict=True)\n    net.cuda().eval()\n    for p in net.parameters():\n        p.requires_grad = False\n    return net, a, ckpt_path, ckpt\n\ndef recover_xyz(motion_norm, mean_np, std_np):\n    arr = motion_norm.detach().float().cpu().numpy()\n    arr = arr * std_np + mean_np\n    xyz = recover_from_ric(torch.from_numpy(arr).float().unsqueeze(0), 22)\n    return xyz.squeeze(0).cpu().numpy()\n\ndef angle_deg(xyz, triplet):\n    a, b, c = triplet\n    v1 = xyz[:, c] - xyz[:, b]\n    v2 = xyz[:, a] - xyz[:, b]\n    v1 = v1 / np.clip(np.linalg.norm(v1, axis=1, keepdims=True), 1e-8, None)\n    v2 = v2 / np.clip(np.linalg.norm(v2, axis=1, keepdims=True), 1e-8, None)\n    cosv = np.clip((v1 * v2).sum(axis=1), -1.0, 1.0)\n    return np.degrees(np.arccos(cosv))\n\ndef angle_category(angles):\n    return np.digitize(angles, ANGLE_THRESHOLDS, right=True).astype(np.int64)\n\ndef edit_pose_tokens(pose_tokens, group_id, tok_start, tok_end, target_cat):\n    out = pose_tokens.copy()\n    g_end, g_start = vq_to_range[group_id]\n    assert (g_end - g_start + 1) == 18\n    out[tok_start:tok_end, g_start:g_end+1] = 0.0\n    out[tok_start:tok_end, g_start + target_cat] = 1.0\n    return out\n\ndef mpjpe_mm(a, b, frame_mask=None, joint_ids=None):\n    if joint_ids is None:\n        joint_ids = np.arange(a.shape[1])\n    if frame_mask is None:\n        frame_mask = np.ones(a.shape[0], dtype=bool)\n    aa = a[frame_mask][:, joint_ids]\n    bb = b[frame_mask][:, joint_ids]\n    if aa.size == 0:\n        return float("nan")\n    return float(np.linalg.norm(aa - bb, axis=-1).mean() * 1000.0)\n\ndef bootstrap_mean_ci(values, B=1000, seed=2027):\n    x = np.asarray(values, dtype=np.float64)\n    x = x[np.isfinite(x)]\n    if len(x) == 0:\n        return np.nan, np.nan, np.nan\n    rng = np.random.default_rng(seed)\n    means = np.empty(B, dtype=np.float64)\n    for i in range(B):\n        means[i] = rng.choice(x, size=len(x), replace=True).mean()\n    return float(x.mean()), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--n-samples", type=int, default=400)\nparser.add_argument("--seed", type=int, default=123)\nparser.add_argument("--tokenizer-dir", type=str, required=True)\nparser.add_argument("--data-root", type=str, required=True)\nparser.add_argument("--exp05-rows", type=str, required=True)\nparser.add_argument("--out-dir", type=str, required=True)\nparser.add_argument("--bootstrap", type=int, default=1000)\nargs = parser.parse_args()\n\nfixseed(args.seed)\nDATA = Path(args.data_root)\nOUT = Path(args.out_dir)\nOUT.mkdir(parents=True, exist_ok=True)\n\nmean_np = np.load(DATA / "Mean.npy").astype(np.float32)\nstd_np = np.load(DATA / "Std.npy").astype(np.float32)\n\nold = pd.read_csv(args.exp05_rows)\nif set(old["strategy"].unique()) != {"pose_only", "retain_residual"}:\n    raise RuntimeError(f"Unexpected Exp.05 strategies: {old[\'strategy\'].unique()}")\n\n# One canonical row per sample; use pose_only only to avoid duplicates.\nprotocol = old[old.strategy == "pose_only"].copy()\nprotocol = protocol.sort_values("sample_index").reset_index(drop=True)\n\nif len(protocol) < args.n_samples:\n    raise RuntimeError(f"Exp.05 has only {len(protocol)} canonical samples")\nprotocol = protocol.iloc[:args.n_samples].copy()\n\nnet, net_args, tokenizer_ckpt, tokenizer_state = build_tokenizer(args.tokenizer_dir)\nprint("[OK] Tokenizer loaded:", tokenizer_ckpt)\nprint("[OK] Tokenizer checkpoint iter:", tokenizer_state.get("nb_iter"))\nprint("[OK] Tokenizer SHA256:", sha256(tokenizer_ckpt))\n\nrows = []\nt0_all = time.time()\n\nfor i, pr in protocol.iterrows():\n    sid = str(pr["sample_id"])\n    target_name = str(pr["target_name"])\n    target = TARGETS[target_name]\n\n    motion_raw = np.load(DATA / "new_joint_vecs" / f"{sid}.npy").astype(np.float32)\n    pose_full = np.load(DATA / "codes" / f"{sid}.npy").astype(np.float32)\n\n    L = int(pr["n_frames"])\n    L = min(L, len(motion_raw), len(pose_full))\n    L = (L // 4) * 4\n\n    motion_raw = motion_raw[:L]\n    pose_tokens = pose_full[:L:4]\n\n    tok_start = int(pr["tok_start"])\n    tok_end = int(pr["tok_end"])\n    target_cat = int(pr["target_cat"])\n    source_med = int(pr["source_median_cat"])\n\n    if tok_end > len(pose_tokens):\n        raise RuntimeError(f"{sid}: Exp.05 window exceeds pose token length")\n\n    edited_tokens = edit_pose_tokens(\n        pose_tokens, target["group_id"], tok_start, tok_end, target_cat\n    )\n\n    motion_norm = ((motion_raw - mean_np) / std_np).astype(np.float32)\n\n    motion_t = torch.from_numpy(motion_norm).float().unsqueeze(0).cuda()\n    pose_t = torch.from_numpy(pose_tokens).float().unsqueeze(0).cuda()\n    edited_t = torch.from_numpy(edited_tokens).float().unsqueeze(0).cuda()\n\n    with torch.no_grad():\n        # Standard unedited full reconstruction.\n        _, original_residual = net.tokenize(\n            motion=motion_t,\n            code_indices=pose_t,\n            return_one_hot=True,\n        )\n        base_norm = net.inference(\n            residual_codes=original_residual,\n            code_indices=pose_t,\n        )\n\n        # Re-encode the residual relative to the EDITED pose latent.\n        _, reencoded_residual = net.tokenize(\n            motion=motion_t,\n            code_indices=edited_t,\n            return_one_hot=True,\n        )\n        edit_norm = net.inference(\n            residual_codes=reencoded_residual,\n            code_indices=edited_t,\n        )\n\n    gt_xyz = recover_xyz(torch.from_numpy(motion_norm), mean_np, std_np)\n    base_xyz = recover_xyz(base_norm[0], mean_np, std_np)\n    edit_xyz = recover_xyz(edit_norm[0], mean_np, std_np)\n\n    T = min(len(gt_xyz), len(base_xyz), len(edit_xyz), L)\n    gt_xyz = gt_xyz[:T]\n    base_xyz = base_xyz[:T]\n    edit_xyz = edit_xyz[:T]\n\n    fs = min(int(pr["frame_start"]), T)\n    fe = min(int(pr["frame_end"]), T)\n    inside = np.zeros(T, dtype=bool)\n    inside[fs:fe] = True\n    outside = ~inside\n    if inside.sum() == 0:\n        raise RuntimeError(f"{sid}: empty edit window")\n\n    target_joints = np.asarray(target["target_joints"], dtype=np.int64)\n    nontarget_joints = np.asarray(\n        [j for j in range(22) if j not in set(target_joints.tolist())],\n        dtype=np.int64,\n    )\n\n    base_ang = angle_deg(base_xyz, target["triplet"])\n    edit_ang = angle_deg(edit_xyz, target["triplet"])\n    base_cat = angle_category(base_ang)\n    edit_cat = angle_category(edit_ang)\n\n    base_err = np.abs(base_cat[inside] - target_cat)\n    edit_err = np.abs(edit_cat[inside] - target_cat)\n\n    sign = 1.0 if target_cat > source_med else -1.0\n    signed_progress_deg = sign * float(\n        edit_ang[inside].mean() - base_ang[inside].mean()\n    )\n\n    row = {\n        "sample_id": sid,\n        "sample_index": int(pr["sample_index"]),\n        "strategy": "reencode_residual",\n        "target_name": target_name,\n        "group_id": int(target["group_id"]),\n        "n_frames": T,\n        "n_tokens": len(pose_tokens),\n        "tok_start": tok_start,\n        "tok_end": tok_end,\n        "frame_start": fs,\n        "frame_end": fe,\n        "source_median_cat": source_med,\n        "target_cat": target_cat,\n        "target_description": str(pr["target_description"]),\n        "source_target_cat_gap": int(pr["source_target_cat_gap"]),\n        "strict_success_rate": float(np.mean(edit_cat[inside] == target_cat)),\n        "toward_target_rate": float(np.mean(edit_err < base_err)),\n        "not_worse_rate": float(np.mean(edit_err <= base_err)),\n        "category_error_reduction": float(base_err.mean() - edit_err.mean()),\n        "signed_angle_progress_deg": signed_progress_deg,\n        "direction_success": float(signed_progress_deg > 0),\n        "outside_window_mpjpe_mm": mpjpe_mm(edit_xyz, base_xyz, outside, None),\n        "inside_nontarget_mpjpe_mm": mpjpe_mm(edit_xyz, base_xyz, inside, nontarget_joints),\n        "inside_target_mpjpe_mm": mpjpe_mm(edit_xyz, base_xyz, inside, target_joints),\n        "locality_ratio": float(\n            mpjpe_mm(edit_xyz, base_xyz, inside, target_joints)\n            / max(mpjpe_mm(edit_xyz, base_xyz, inside, nontarget_joints), 1e-8)\n        ),\n        "baseline_recon_mpjpe_mm": mpjpe_mm(base_xyz, gt_xyz),\n    }\n    rows.append(row)\n\n    if (len(rows) % 10 == 0) or (len(rows) == args.n_samples):\n        pd.DataFrame(rows).to_csv(OUT / "exp05b_reencode_partial.csv", index=False)\n        elapsed = (time.time() - t0_all) / 60.0\n        print(\n            f"[RUN] {len(rows)}/{args.n_samples} motions "\n            f"({100*len(rows)/args.n_samples:.1f}%) elapsed={elapsed:.1f} min",\n            flush=True,\n        )\n\nnew = pd.DataFrame(rows)\nif len(new) != args.n_samples:\n    raise RuntimeError(f"Expected {args.n_samples} rows, got {len(new)}")\n\nnew.to_csv(OUT / "exp05b_reencode_per_sample.csv", index=False)\n\n# Audit that the unedited baseline matches Exp.05 retain_residual closely.\nold_ret = old[old.strategy == "retain_residual"][\n    ["sample_id", "baseline_recon_mpjpe_mm"]\n].copy()\nchk = new[["sample_id", "baseline_recon_mpjpe_mm"]].merge(\n    old_ret, on="sample_id", suffixes=("_new", "_exp05")\n)\nbaseline_abs_diff = np.abs(\n    chk["baseline_recon_mpjpe_mm_new"] - chk["baseline_recon_mpjpe_mm_exp05"]\n)\nbaseline_max_abs_diff = float(baseline_abs_diff.max())\nbaseline_mean_abs_diff = float(baseline_abs_diff.mean())\n\nprint(\n    "[AUDIT] baseline reconstruction diff vs Exp.05 retain_residual:",\n    "mean_abs=", baseline_mean_abs_diff,\n    "max_abs=", baseline_max_abs_diff,\n)\n\n# Combine all three strategies.\nold_keep = old[old["sample_id"].isin(new["sample_id"])].copy()\ncombined = pd.concat([old_keep, new], ignore_index=True)\ncombined.to_csv(OUT / "exp05b_all_three_per_sample.csv", index=False)\n\nMETRICS = [\n    "strict_success_rate",\n    "toward_target_rate",\n    "not_worse_rate",\n    "category_error_reduction",\n    "signed_angle_progress_deg",\n    "direction_success",\n    "outside_window_mpjpe_mm",\n    "inside_nontarget_mpjpe_mm",\n    "inside_target_mpjpe_mm",\n    "locality_ratio",\n    "baseline_recon_mpjpe_mm",\n]\n\nstrategies = ["pose_only", "retain_residual", "reencode_residual"]\n\nsummary_rows = []\nfor strategy in strategies:\n    d = combined[combined.strategy == strategy]\n    row = {"strategy": strategy, "n": len(d)}\n    for mi, metric in enumerate(METRICS):\n        mean, lo, hi = bootstrap_mean_ci(\n            d[metric].to_numpy(),\n            B=args.bootstrap,\n            seed=2027 + mi,\n        )\n        row[f"{metric}_mean"] = mean\n        row[f"{metric}_ci95_lo"] = lo\n        row[f"{metric}_ci95_hi"] = hi\n    summary_rows.append(row)\n\nsummary = pd.DataFrame(summary_rows)\nsummary.to_csv(OUT / "exp05b_three_policy_summary.csv", index=False)\n\nby_target_rows = []\nfor strategy in strategies:\n    for target_name in TARGETS:\n        d = combined[\n            (combined.strategy == strategy)\n            & (combined.target_name == target_name)\n        ]\n        row = {"strategy": strategy, "target_name": target_name, "n": len(d)}\n        for mi, metric in enumerate(METRICS):\n            mean, lo, hi = bootstrap_mean_ci(\n                d[metric].to_numpy(),\n                B=args.bootstrap,\n                seed=3027 + mi,\n            )\n            row[f"{metric}_mean"] = mean\n            row[f"{metric}_ci95_lo"] = lo\n            row[f"{metric}_ci95_hi"] = hi\n        by_target_rows.append(row)\n\nby_target = pd.DataFrame(by_target_rows)\nby_target.to_csv(OUT / "exp05b_three_policy_by_target.csv", index=False)\n\n# Paired comparisons.\nwide = combined.pivot(index="sample_id", columns="strategy", values=METRICS)\n\npairs = [\n    ("retain_residual", "pose_only"),\n    ("reencode_residual", "pose_only"),\n    ("reencode_residual", "retain_residual"),\n]\n\npaired_rows = []\nfor a, b in pairs:\n    for mi, metric in enumerate(METRICS):\n        x = (wide[(metric, a)] - wide[(metric, b)]).to_numpy()\n        mean, lo, hi = bootstrap_mean_ci(\n            x,\n            B=args.bootstrap,\n            seed=4027 + mi + 100 * pairs.index((a,b)),\n        )\n        paired_rows.append({\n            "metric": metric,\n            "comparison": f"{a} - {b}",\n            "mean_diff": mean,\n            "ci95_lo": lo,\n            "ci95_hi": hi,\n        })\n\npaired = pd.DataFrame(paired_rows)\npaired.to_csv(OUT / "exp05b_three_policy_paired.csv", index=False)\n\naudit = {\n    "experiment": "Exp05B residual re-encoding policy",\n    "n_samples": args.n_samples,\n    "seed": args.seed,\n    "source_exp05_rows": str(args.exp05_rows),\n    "source_exp05_rows_sha256": sha256(args.exp05_rows),\n    "tokenizer_checkpoint": str(tokenizer_ckpt),\n    "tokenizer_sha256": sha256(tokenizer_ckpt),\n    "tokenizer_checkpoint_iter": tokenizer_state.get("nb_iter"),\n    "strategies": strategies,\n    "baseline_recon_mean_abs_diff_vs_exp05_retain": baseline_mean_abs_diff,\n    "baseline_recon_max_abs_diff_vs_exp05_retain": baseline_max_abs_diff,\n    "elapsed_minutes": (time.time() - t0_all) / 60.0,\n    "outputs": {\n        "reencode_per_sample": str(OUT / "exp05b_reencode_per_sample.csv"),\n        "all_three_per_sample": str(OUT / "exp05b_all_three_per_sample.csv"),\n        "summary": str(OUT / "exp05b_three_policy_summary.csv"),\n        "by_target": str(OUT / "exp05b_three_policy_by_target.csv"),\n        "paired": str(OUT / "exp05b_three_policy_paired.csv"),\n    },\n}\n(OUT / "exp05b_completion_audit.json").write_text(json.dumps(audit, indent=2))\n\nprint("\\n=== EXP.05B THREE-POLICY SUMMARY ===")\nprint(summary.to_string(index=False))\n\nprint("\\n=== EXP.05B PAIRED POLICY DIFFERENCES ===")\nprint(paired.to_string(index=False))\n\nprint("\\n[OK] EXP05B_COMPLETE")\n'
SCRIPT.write_text(script_text)
compile(SCRIPT.read_text(), str(SCRIPT), 'exec')
print('[OK] Script:', SCRIPT)


In [ ]:

# Smoke test: reuse the first 8 motions from Exp.05 exactly.

SMOKE = RESULTS / "smoke"
SMOKE.mkdir(parents=True, exist_ok=True)
SMOKE_LOG = LOGDIR / "exp05b_smoke.log"

cmd = [
    str(PY), "-u", str(SCRIPT),
    "--n-samples", "8",
    "--seed", "123",
    "--tokenizer-dir", str(TOKENIZER_DIR),
    "--data-root", str(DATA),
    "--exp05-rows", str(EXP05_ROWS),
    "--out-dir", str(SMOKE),
    "--bootstrap", "200",
]

print("=== EXP.05B SMOKE ===")
print(" ".join(shlex.quote(x) for x in cmd))

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

t0 = time.time()
with open(SMOKE_LOG, "w") as f:
    p = subprocess.run(
        cmd,
        cwd=PGR,
        env=env,
        stdout=f,
        stderr=subprocess.STDOUT,
        timeout=7200,
    )

elapsed = (time.time() - t0) / 60
txt = SMOKE_LOG.read_text(errors="replace")

print("returncode:", p.returncode)
print(f"elapsed: {elapsed:.1f} min")
print("\n".join(txt.splitlines()[-180:]))

if p.returncode != 0:
    raise RuntimeError(f"Exp.05B smoke failed: {SMOKE_LOG}")
if "[OK] EXP05B_COMPLETE" not in txt:
    raise RuntimeError("Exp.05B smoke returned 0 but completion marker is missing.")

print("[OK] Exp.05B end-to-end smoke passed.")



## Production run

Run the next cell only after the smoke test prints:

```text
[OK] Exp.05B end-to-end smoke passed.
```

This production run computes **only the new `reencode_residual` policy** for the same
400 Exp.05 motions, then merges it with the already-completed Exp.05 results.

It does **not** rerun `pose_only` or `retain_residual`.


In [ ]:

# Launch 400-motion Exp.05B in the background.

PROD = RESULTS / "production"
PROD.mkdir(parents=True, exist_ok=True)

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

old_pid = None
if PIDFILE.exists():
    try:
        old_pid = int(PIDFILE.read_text().strip())
    except Exception:
        pass

old_txt = LOG.read_text(errors="replace") if LOG.exists() else ""

if "[OK] EXP05B_COMPLETE" in old_txt:
    print("=== EXP.05B ALREADY COMPLETE ===")
elif old_pid and pid_alive(old_pid):
    print("=== EXP.05B ALREADY RUNNING ===")
    print("PID:", old_pid)
else:
    cmd = [
        str(PY), "-u", str(SCRIPT),
        "--n-samples", "400",
        "--seed", "123",
        "--tokenizer-dir", str(TOKENIZER_DIR),
        "--data-root", str(DATA),
        "--exp05-rows", str(EXP05_ROWS),
        "--out-dir", str(PROD),
        "--bootstrap", "1000",
    ]

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"

    handle = open(LOG, "w")
    p = subprocess.Popen(
        cmd,
        cwd=PGR,
        env=env,
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    PIDFILE.write_text(str(p.pid))
    LAUNCH.write_text(json.dumps({
        "pid": p.pid,
        "command": cmd,
        "log": str(LOG),
        "out_dir": str(PROD),
        "launched_at_unix": time.time(),
    }, indent=2))

    time.sleep(1)
    print("=== EXP.05B LAUNCH ===")
    print("PID   :", p.pid)
    print("Alive :", pid_alive(p.pid))
    print("Log   :", LOG)
    print("Out   :", PROD)


In [ ]:

# Read-only status cell. Safe to rerun.

import pandas as pd

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

pid = None
if PIDFILE.exists():
    try:
        pid = int(PIDFILE.read_text().strip())
    except Exception:
        pass

partial = RESULTS / "production" / "exp05b_reencode_partial.csv"
done = 0
if partial.exists():
    d = pd.read_csv(partial)
    done = len(d)

txt = LOG.read_text(errors="replace") if LOG.exists() else ""
complete = "[OK] EXP05B_COMPLETE" in txt

print("=== EXP.05B STATUS ===")
print("PID      :", pid)
print("Alive    :", pid_alive(pid) if pid else False)
print("Motions  :", f"{done}/400")
print("Progress :", f"{100*done/400:.1f}%")
print("Complete :", complete)

print("\n=== LOG TAIL ===")
print("\n".join(txt.splitlines()[-120:]))


In [ ]:

# Final reviewer-ready tables. Run after Complete=True.

import json
import pandas as pd

PROD = RESULTS / "production"
AUDIT = PROD / "exp05b_completion_audit.json"
if not AUDIT.exists():
    raise RuntimeError("Exp.05B production is not complete yet.")

audit = json.loads(AUDIT.read_text())
summary = pd.read_csv(PROD / "exp05b_three_policy_summary.csv")
by_target = pd.read_csv(PROD / "exp05b_three_policy_by_target.csv")
paired = pd.read_csv(PROD / "exp05b_three_policy_paired.csv")

print("=== EXP.05B COMPLETION AUDIT ===")
print(json.dumps(audit, indent=2))

primary_cols = [
    "strategy", "n",
    "strict_success_rate_mean",
    "toward_target_rate_mean",
    "category_error_reduction_mean",
    "signed_angle_progress_deg_mean",
    "outside_window_mpjpe_mm_mean",
    "inside_nontarget_mpjpe_mm_mean",
    "inside_target_mpjpe_mm_mean",
    "locality_ratio_mean",
    "baseline_recon_mpjpe_mm_mean",
]

print("\n=== THREE-POLICY PRIMARY SUMMARY ===")
display(summary[primary_cols].round(5))

print("\n=== THREE-POLICY BY TARGET ===")
display(by_target[
    [
        "strategy", "target_name", "n",
        "strict_success_rate_mean",
        "toward_target_rate_mean",
        "signed_angle_progress_deg_mean",
        "outside_window_mpjpe_mm_mean",
        "inside_nontarget_mpjpe_mm_mean",
        "inside_target_mpjpe_mm_mean",
        "locality_ratio_mean",
    ]
].round(5))

print("\n=== PAIRED POLICY DIFFERENCES ===")
display(paired.round(5))

print("\n[OK] Exp.05B reviewer tables ready.")



## Interpretation guide

The central reviewer-facing comparison is:

| Policy | Expected behavior |
|---|---|
| pose-only | strongest edit, weakest preservation |
| retain residual | preservation/edit-strength compromise |
| re-encode residual | may reconstruct the original motion and suppress the edit |

For **edit effectiveness**, higher is better:
- toward-target rate
- category-error reduction
- signed angle progress
- strict target-category success

For **preservation**, lower is better:
- outside-window MPJPE
- non-target-joint MPJPE

For **locality**, a larger target/non-target edit ratio indicates that motion changes are
more concentrated in the intended body part.

If re-encoding gives excellent preservation but substantially lower edit progress, that is
direct evidence that re-encoding the residual can absorb/cancel the pose-code intervention.
In that case, `retain_residual` is the most defensible practical compromise.

Do not describe this as a universal optimality result; it is a controlled inference-policy
study using the fixed trained tokenizer and four interpretable angle edits.
